In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import seaborn as sns
import os
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, month_plot
from scipy.stats import boxcox 
from statsmodels.tsa.seasonal import STL
from utils import load_data, check_stationarity
from plots import *

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Load & Process Data

In [ ]:
data = load_data()

In [ ]:
data["t_seasdiff"]= data["tdiff"].diff(12)
data_sd = data.dropna(subset=["t_seasdiff"])

## Plot time Series

In [ ]:
data

In [ ]:
plot_time_series(data, 'date', 'tdiff', ma_window=12)

In [ ]:
plot_time_series(data, 'date', 'tmed', ma_window=12)

In [ ]:
plot_time_series(data, 'date', 'tmax', ma_window=12)

In [ ]:
plot_time_series(data, 'date', 'tmin', ma_window=12)

In [ ]:
plot_time_series(data, 'date', 'prec', ma_window=12)

### Log Feat

In [ ]:
plot_time_series(data, 'date', 'log_diff', ma_window=12)

In [ ]:
plot_time_series(data_sd, 'date', 't_seasdiff', ma_window=12)

## Seasonal Plot

In [ ]:
seasonal_plot(data, 'tdiff', "Seasonal Subseries Plot: Temperature Range (Max - Min)")

In [ ]:
seasonal_plot(data, 'log_diff', "Seasonal Subseries Plot: Log Temperature Range (Max - Min)")

## Box Cox

In [ ]:
bc, lambda_ = boxcox(data['tdiff'])
print(f"Estimated Lambda: {lambda_}")

plot_time_series(pd.DataFrame.from_dict({
    "date": list(data.index),
    "box_cox": bc
}), 'date', 'box_cox', ma_window=12)

In [ ]:
bc, lambda_ = boxcox(data['log_diff'])
print(f"Estimated Lambda: {lambda_}")

plot_time_series(pd.DataFrame.from_dict({
    "date": list(data.index),
    "box_cox": bc
}), 'date', 'box_cox', ma_window=12)

## Linear Regression

In [ ]:
import statsmodels.formula.api as smf 
data['time'] = data['date'].dt.to_period('M').astype(int) 

# fir linear regression model
fit = smf.ols(formula='tdiff ~ time', data=data).fit()
print(fit.summary())

In [ ]:
residuals = fit.resid
# Resíduos: detrended time series
plt.subplot(1, 1, 1)
plt.scatter(data['date'], residuals, color='blue')
plt.title("Temperature Range Detrended")
plt.ylabel("Residuals")
plt.xlabel("Time")

plt.show()

In [ ]:
import statsmodels.formula.api as smf 

data['time'] = data['date'].dt.to_period('M').astype(int) 

# fir linear regression model
fit = smf.ols(formula='tdiff ~ time', data=data).fit()
print(fit.summary())

residuals = fit.resid
# Resíduos: detrended time series
plt.subplot(1, 1, 1)
plt.plot(data['date'], residuals, color='blue')
plt.title("Temperature Range Detrended")
plt.ylabel("Residuals")
plt.xlabel("Time")

plt.show()

## Difference Operator

In [ ]:
tdiff_series = data['tdiff']
tdiff_diff = tdiff_series.diff(1)
tdiff_diff

In [ ]:
plt.plot(data.index, tdiff_diff, color='#ff9900')
plt.title("Temperature Range Differenced")
plt.ylabel("Temperature Range")
plt.xlabel("date")

In [ ]:
diff12 = tdiff_diff.to_frame()
diff12['year_month'] = pd.to_datetime(diff12.index)
diff12

In [ ]:
diff12 = tdiff_diff.to_frame()
diff12['year_month'] = pd.to_datetime(diff12.index)

x = diff12[['year_month','tdiff']]
x = x.set_index('year_month')

seasonal_plot(x, 'tdiff', "Seasonal Plot: tdiff diff 12")


## STL Decomposition

In [ ]:
# STL with periodic seasonal component
stl_1 = STL(data['tdiff'], seasonal=len(data), robust=True).fit()

# STL with seasonal window of 13
stl_2 = STL(data['tdiff'], seasonal=13, robust=True).fit()

In [ ]:
print("Periodic Season")
stl_1.plot()

In [ ]:
print("13 Season")
stl_2.plot()

In [ ]:
from statsmodels.stats.diagnostic import het_arch
print(het_arch(stl_1.resid)) # Not heteroskedascity, null would be homoskedascity, and p-value >> 0.05

In [ ]:
print(het_arch(stl_1.resid, nlags=12)) 

## Lag Plots

In [ ]:
lag_plot_grid(data['tdiff'], ys=data['prec'], title="Lag Plots: Tdiff v Prec")

In [ ]:
lag_plot_grid(data['tdiff'], ys=data['tmax'], title="Lag Plots: Tdiff v Tmax")

In [ ]:
lag_plot_grid(data['tdiff'], ys=data['tmin'], title="Lag Plots: Tdiff v Tmin")

In [ ]:
lag_plot_grid(data['tdiff'], title="Lag Plots: Tdiff")

In [ ]:
lag_plot_grid(data['log_diff'], title="Lag Plots: Log Tdiff")
lag_plot_grid(stl_1.resid, title="Lag Plots: Log Tdiff - Remainder")

## ACF

In [ ]:
plot_acfs(data, 'prec')

In [ ]:
plot_acfs(data, 'tdiff')

In [ ]:
data["t_diff"]= data["tdiff"].diff(12)
data_diff = data.dropna(subset=["t_diff"])
plot_acfs(data_diff, 't_diff')

In [ ]:
plot_acfs(data_sd, 't_seasdiff')

## Stationarity

In [ ]:
check_stationarity(data['tdiff'])

In [ ]:
check_stationarity(data['tdiff'].diff().dropna())

In [ ]:
check_stationarity(data_sd['t_seasdiff'])